# Week 3 — Command line, shells, and the terminal

**Notebook outcomes**

- Navigate the filesystem with `cd`, `ls`, `pwd`
- Read and search files with `cat`, `less`, `grep`, `head`, `tail`
- Combine commands with pipes and redirection
- Understand environment variables and shell configuration
- Preview `ssh` for working on remote machines


## Why the terminal?

You already ran shell commands in week 1 (`uv`, `curl`). Here we slow down
and learn the shell itself. A comfortable shell user is dramatically
faster at everyday tasks — file wrangling, running scripts, managing
servers, debugging.

For this camp we'll use a POSIX-ish shell: **zsh** on macOS, **bash** on
most Linux. Nearly everything below works identically in both. On
Windows, use **WSL** (Windows Subsystem for Linux) or **Git Bash**.


## Shell, terminal, prompt — what's what?

Confusing set of terms:

- **Terminal (emulator)** — the *app* (Terminal.app, Alacritty, Windows
  Terminal). It draws text to a window.
- **Shell** — the *program* running inside the terminal that interprets
  commands (zsh, bash, fish).
- **Prompt** — the text the shell prints before it waits for input
  (usually ends in `$` or `%`).

In this notebook, the `!` prefix on a cell tells Jupyter to send the rest
of the line to the shell. You can follow along without leaving the
notebook — but do also open a real terminal to build muscle memory.


## Where am I? Where are my files?

Three commands you'll type hundreds of times per week:


In [ ]:
!pwd       # print working directory

In [ ]:
!ls        # list the current directory

In [ ]:
!ls -la    # long format, including hidden (dot-)files

Move around with `cd`:

```sh
cd ~              # your home directory
cd /tmp           # absolute path
cd ../other       # relative: parent, then into `other`
cd -              # toggle back to the previous directory
```

`~` is shorthand for your home directory. `.` means "here". `..` means
"parent".


## Peeking at files

```sh
cat  file.txt        # print the whole file
less file.txt        # scrollable pager — q to quit
head -n 5 file.txt   # first 5 lines
tail -n 5 file.txt   # last 5 lines
tail -f log.txt      # follow new lines as they append (Ctrl+C to stop)
```

Let's try one — read the first few lines of a file that's guaranteed to
exist on any Unix:


In [ ]:
!head -n 3 /etc/hosts

## Pipes

A **pipe** (`|`) sends the output of one command into the input of the
next. This is the single most important shell idea.

```sh
ls | wc -l          # how many entries in the current directory?
```

- `ls` prints filenames.
- `wc -l` counts lines.
- Together: "how many files".


In [ ]:
!ls /usr/bin | wc -l

## Searching with `grep`

`grep PATTERN FILE` prints lines in `FILE` that match `PATTERN`.

```sh
grep TODO main.py              # lines with TODO
grep -n TODO main.py           # include line numbers
grep -i error log.txt          # case-insensitive
grep -r "def foo" src/         # recursive search under a directory
```

`grep` shines in pipes:


In [ ]:
!ls /usr/bin | grep -i python

## Redirection

`>` redirects a command's **output** to a file (overwriting). `>>`
**appends**. `<` feeds a file in as the command's **input**.

```sh
echo "hello" >  greeting.txt    # write
echo "world" >> greeting.txt    # append
wc -l < greeting.txt            # read
```

`2>` redirects **errors** specifically; `2>&1` merges errors into stdout.

```sh
some_command > out.log 2> err.log
some_command > all.log 2>&1
```


## Wildcards (globbing)

The shell expands these **before** the command runs.

| Glob | Matches |
|------|---------|
| `*.py` | any filename ending in `.py` |
| `L??_*` | any filename starting with `L` + two chars + `_` + anything |
| `lectures/**/*.ipynb` | any `.ipynb` anywhere under `lectures/` (zsh; on bash needs `shopt -s globstar`) |


In [ ]:
!ls /home/chase/teaching/Rice/computational_camp/lectures/L0[1-3]_*/ 2>/dev/null || ls /tmp | head -3

## Environment variables

The shell keeps a set of **environment variables** — key-value pairs
shared with every program it runs. See yours with:

```sh
env | less
```

A few important ones:

| Variable | What |
|----------|------|
| `PATH` | colon-separated list of directories the shell searches for commands |
| `HOME` | your home directory |
| `SHELL` | which shell is running |
| `EDITOR` | which editor tools like `git` use |

Read one:


In [ ]:
!echo "$HOME"

Set one for a single command:

```sh
EDITOR=nano git commit
```

Set one for the rest of the session:

```sh
export EDITOR=code
```

Make it permanent by adding `export EDITOR=code` to `~/.zshrc` (zsh) or
`~/.bashrc` (bash).


## Shell configuration files

Every time a new interactive shell starts, it reads a config file:

- **zsh** reads `~/.zshrc`
- **bash** reads `~/.bashrc` (interactive) or `~/.bash_profile` (login)

Common things to put there:

```sh
# prettier prompt
PS1='%n@%m %~ %# '

# shortcuts (aliases)
alias ll='ls -la'
alias gs='git status'

# environment
export EDITOR=code
export PATH="$HOME/.local/bin:$PATH"
```

After editing, run `source ~/.zshrc` (or open a new terminal) to reload.


## Pipelines in practice

A small pipeline combines everything so far:

```sh
# 10 most common words in a file
cat file.txt | tr ' ' '\n' | sort | uniq -c | sort -rn | head
```

Read it left to right:

1. `cat file.txt` — emit the file.
2. `tr ' ' '\n'` — replace spaces with newlines (one word per line).
3. `sort` — alphabetize.
4. `uniq -c` — collapse duplicates, prefix with count.
5. `sort -rn` — sort by count, descending.
6. `head` — top 10.

This is the essence of shell thinking: small tools, chained.


## Working on another machine: `ssh`

Eventually you'll want to run code on a server — a Rice cluster, a cloud
VM, a shared GPU machine. The door in is `ssh`:

```sh
ssh netid@server.rice.edu
```

Once connected, you're in a shell on the remote machine. All the commands
above work there too. `exit` or `Ctrl+D` disconnects.

Copy files to/from with `scp` or (better) `rsync`:

```sh
scp local_file.py netid@server:/path/on/server/
rsync -av results/ netid@server:~/project/results/
```

Full SSH/cluster setup is beyond scope for the camp — we'll point you at
the Rice research computing docs when you need it.


## Cheat sheet

```sh
pwd                 # where am I?
ls [-la]            # what's here?
cd PATH             # go there
cat / less / head / tail FILE
grep PATTERN FILE   # search
|                   # pipe
>  / >> / <         # redirect
*  / ?              # glob
$VAR / export VAR=  # env vars
~/.zshrc            # per-user config
ssh user@host       # remote shell
```

You won't remember all of this today. That's fine. The goal is to
recognize the shapes so that when you see them online you don't freeze.
